# Atividade Avaliativa — MapReduce
## Análise de viagens de táxi de Nova York — 2024

**Processamento de Dados Massivos**

Este notebook responde às 6 questões da atividade utilizando explicitamente o paradigma:

**MAP → SHUFFLE → REDUCE**

A implementação segue a ideia do notebook apresentado em sala, trabalhando com arquivos intermediários e funções Python. Para a etapa MapReduce não é utilizado `pandas`.

> O arquivo utilizado neste notebook é `nyc_tripdata_2024_sample_1M.csv`.

https://huggingface.co/datasets/alexvaroz/nyc_taxi_trip_2024_p1_sample/resolve/main/nyc_tripdata_2024_sample_1M.csv

## 1. Preparação e inspeção do arquivo

Primeiro verificamos o cabeçalho e a quantidade de registros. A biblioteca `csv` faz parte da biblioteca padrão do Python e é usada apenas para interpretar corretamente as linhas do CSV.

In [1]:
from google.colab import files

uploaded = files.upload()

ARQUIVO = next(iter(uploaded))

print("Arquivo selecionado:", ARQUIVO)

Saving nyc_tripdata_2024_sample_1M.csv to nyc_tripdata_2024_sample_1M.csv
Arquivo selecionado: nyc_tripdata_2024_sample_1M.csv


In [2]:
import csv
import ast
from collections import defaultdict
from functools import reduce
from datetime import datetime

#ARQUIVO = "nyc_tripdata_2024_sample_1M.csv"

with open(ARQUIVO, "r", encoding="utf-8", newline="") as arquivo:
    leitor = csv.reader(arquivo)
    cabecalho = next(leitor)
    quantidade_registros = sum(1 for _ in leitor)

print("Quantidade de registros:", quantidade_registros)
print("Colunas:")
for i, coluna in enumerate(cabecalho):
    print(i, coluna)

Quantidade de registros: 1000000
Colunas:
0 VendorID
1 tpep_pickup_datetime
2 tpep_dropoff_datetime
3 passenger_count
4 trip_distance
5 RatecodeID
6 store_and_fwd_flag
7 PULocationID
8 DOLocationID
9 payment_type
10 fare_amount
11 extra
12 mta_tax
13 tip_amount
14 tolls_amount
15 improvement_surcharge
16 total_amount
17 congestion_surcharge
18 Airport_fee
19 arquivo_origem


### Campos do dicionário de dados usados na atividade

- `payment_type`: código da forma de pagamento.
- `fare_amount`: tarifa calculada pelo taxímetro.
- `total_amount`: valor total cobrado do passageiro.
- `trip_distance`: distância da viagem em milhas.
- `tpep_pickup_datetime`: data e hora em que o taxímetro foi acionado.

Códigos de `payment_type`:

- 0 = Flex Fare trip
- 1 = Credit card
- 2 = Cash
- 3 = No charge
- 4 = Dispute
- 5 = Unknown
- 6 = Voided trip

In [3]:
TIPOS_PAGAMENTO = {
    "0": "Flex Fare trip",
    "1": "Credit card",
    "2": "Cash",
    "3": "No charge",
    "4": "Dispute",
    "5": "Unknown",
    "6": "Voided trip"
}

## 2. Função Shuffle

O *mapper* produz pares `chave → valor`.  
O *shuffle* agrupa todos os valores que possuem a mesma chave.

O arquivo de saída do shuffle é um dicionário no formato:

```text
{
    chave1: [valor1, valor2, ...],
    chave2: [valor1, valor2, ...]
}
```

In [4]:
def shuffle(nome_arquivo_entrada, nome_arquivo_saida):
    resultado_intermediario = defaultdict(list)

    with open(nome_arquivo_entrada, "r", encoding="utf-8") as arquivo_entrada:
        for linha in arquivo_entrada:
            linha = linha.rstrip("\n")
            if not linha:
                continue

            chave, valor = linha.split("\t", 1)
            resultado_intermediario[chave].append(valor)

    with open(nome_arquivo_saida, "w", encoding="utf-8") as arquivo_shuffle:
        arquivo_shuffle.write(repr(dict(resultado_intermediario)))

# Questão 1 — Número de viagens por tipo de pagamento

### MAP
Para cada viagem emitimos:

```text
tipo_pagamento    1
```

### SHUFFLE
Agrupamos todos os `1` pelo tipo de pagamento.

### REDUCE
Somamos os valores de cada grupo.

In [5]:
def map_viagens_por_pagamento(nome_arquivo_entrada, nome_arquivo_saida):
    with open(nome_arquivo_entrada, "r", encoding="utf-8", newline="") as entrada, \
         open(nome_arquivo_saida, "w", encoding="utf-8") as saida:

        leitor = csv.DictReader(entrada)

        for linha in leitor:
            codigo = linha["payment_type"].strip()
            tipo = TIPOS_PAGAMENTO.get(codigo, f"Código {codigo}")
            saida.write(f"{tipo}\t1\n")


def reducer_soma_inteiros(nome_arquivo_entrada, nome_arquivo_saida):
    with open(nome_arquivo_entrada, "r", encoding="utf-8") as arquivo:
        grupos = ast.literal_eval(arquivo.read())

    resultado = {}

    for chave, valores in grupos.items():
        resultado[chave] = reduce(lambda x, y: x + y, map(int, valores))

    with open(nome_arquivo_saida, "w", encoding="utf-8") as saida:
        for chave in sorted(resultado):
            saida.write(f"{chave}\t{resultado[chave]}\n")

    return resultado


map_viagens_por_pagamento(ARQUIVO, "q1_mapper.txt")
shuffle("q1_mapper.txt", "q1_shuffle.txt")
resultado_q1 = reducer_soma_inteiros("q1_shuffle.txt", "q1_resultado.txt")

print("Número de viagens por tipo de pagamento:")
for tipo, quantidade in sorted(resultado_q1.items()):
    print(f"{tipo}: {quantidade}")

Número de viagens por tipo de pagamento:
Cash: 136221
Credit card: 743405
Dispute: 16543
Flex Fare trip: 97124
No charge: 6707


# Questão 2 — Receita total por tipo de pagamento

Para representar a **receita total**, utilizamos `total_amount`, que corresponde ao valor total cobrado do passageiro.

### MAP
Emitimos:

```text
tipo_pagamento    total_amount
```

### SHUFFLE
Agrupamos os valores por tipo de pagamento.

### REDUCE
Somamos os valores de cada grupo.

In [6]:
def map_receita_por_pagamento(nome_arquivo_entrada, nome_arquivo_saida):
    with open(nome_arquivo_entrada, "r", encoding="utf-8", newline="") as entrada, \
         open(nome_arquivo_saida, "w", encoding="utf-8") as saida:

        leitor = csv.DictReader(entrada)

        for linha in leitor:
            codigo = linha["payment_type"].strip()
            total = linha["total_amount"].strip()

            if total == "":
                continue

            tipo = TIPOS_PAGAMENTO.get(codigo, f"Código {codigo}")
            saida.write(f"{tipo}\t{float(total)}\n")


def reducer_soma_float(nome_arquivo_entrada, nome_arquivo_saida):
    with open(nome_arquivo_entrada, "r", encoding="utf-8") as arquivo:
        grupos = ast.literal_eval(arquivo.read())

    resultado = {}

    for chave, valores in grupos.items():
        resultado[chave] = reduce(lambda x, y: x + y, map(float, valores))

    with open(nome_arquivo_saida, "w", encoding="utf-8") as saida:
        for chave in sorted(resultado):
            saida.write(f"{chave}\t{resultado[chave]:.2f}\n")

    return resultado


map_receita_por_pagamento(ARQUIVO, "q2_mapper.txt")
shuffle("q2_mapper.txt", "q2_shuffle.txt")
resultado_q2 = reducer_soma_float("q2_shuffle.txt", "q2_resultado.txt")

print("Receita total por tipo de pagamento:")
for tipo, receita in sorted(resultado_q2.items()):
    print(f"{tipo}: $ {receita:.2f}")

Receita total por tipo de pagamento:
Cash: $ 3168095.90
Credit card: $ 21785219.95
Dispute: $ 25214.51
Flex Fare trip: $ 2376069.77
No charge: $ 53932.48


# Questão 3 — Tarifa média cobrada nas viagens

Usamos `fare_amount`, pois o dicionário define esse campo como a tarifa de tempo e distância calculada pelo taxímetro.

Para calcular uma média com MapReduce, o mapper envia cada tarifa para uma chave única (`tarifa`). Depois do shuffle, o reducer calcula:

\[
\text{média} = \frac{\sum tarifas}{n}
\]

In [7]:
def map_tarifas(nome_arquivo_entrada, nome_arquivo_saida):
    with open(nome_arquivo_entrada, "r", encoding="utf-8", newline="") as entrada, \
         open(nome_arquivo_saida, "w", encoding="utf-8") as saida:

        leitor = csv.DictReader(entrada)

        for linha in leitor:
            tarifa = linha["fare_amount"].strip()

            if tarifa == "":
                continue

            saida.write(f"tarifa\t{float(tarifa)}\n")


def reducer_media(nome_arquivo_entrada, nome_arquivo_saida):
    with open(nome_arquivo_entrada, "r", encoding="utf-8") as arquivo:
        grupos = ast.literal_eval(arquivo.read())

    valores = list(map(float, grupos["tarifa"]))
    soma = reduce(lambda x, y: x + y, valores)
    media = soma / len(valores)

    with open(nome_arquivo_saida, "w", encoding="utf-8") as saida:
        saida.write(f"Tarifa média\t{media:.4f}\n")

    return media


map_tarifas(ARQUIVO, "q3_mapper.txt")
shuffle("q3_mapper.txt", "q3_shuffle.txt")
resultado_q3 = reducer_media("q3_shuffle.txt", "q3_resultado.txt")

print(f"Tarifa média cobrada: $ {resultado_q3:.4f}")

Tarifa média cobrada: $ 18.8603


# Questão 4 — Data e hora em que foi feita a viagem mais longa

### MAP
Para cada registro emitimos uma chave única (`viagem`) e, como valor, guardamos:

```text
distância|data_hora_pickup
```

### SHUFFLE
Todas as viagens ficam agrupadas na mesma chave.

### REDUCE
Comparamos as distâncias e mantemos a viagem de maior `trip_distance`.

In [8]:
def map_viagem_mais_longa(nome_arquivo_entrada, nome_arquivo_saida):
    with open(nome_arquivo_entrada, "r", encoding="utf-8", newline="") as entrada, \
         open(nome_arquivo_saida, "w", encoding="utf-8") as saida:

        leitor = csv.DictReader(entrada)

        for linha in leitor:
            distancia = linha["trip_distance"].strip()
            data_hora = linha["tpep_pickup_datetime"].strip()

            if distancia == "" or data_hora == "":
                continue

            saida.write(f"viagem\t{float(distancia)}|{data_hora}\n")


def reducer_viagem_mais_longa(nome_arquivo_entrada, nome_arquivo_saida):
    with open(nome_arquivo_entrada, "r", encoding="utf-8") as arquivo:
        grupos = ast.literal_eval(arquivo.read())

    def maior_viagem(x, y):
        distancia_x = float(x.split("|", 1)[0])
        distancia_y = float(y.split("|", 1)[0])
        return x if distancia_x >= distancia_y else y

    maior = reduce(maior_viagem, grupos["viagem"])
    distancia, data_hora = maior.split("|", 1)

    with open(nome_arquivo_saida, "w", encoding="utf-8") as saida:
        saida.write(f"{data_hora}\t{distancia}\n")

    return data_hora, float(distancia)


map_viagem_mais_longa(ARQUIVO, "q4_mapper.txt")
shuffle("q4_mapper.txt", "q4_shuffle.txt")
data_hora_q4, distancia_q4 = reducer_viagem_mais_longa("q4_shuffle.txt", "q4_resultado.txt")

print("Viagem mais longa:")
print("Data/hora de início:", data_hora_q4)
print(f"Distância: {distancia_q4:.2f} milhas")

Viagem mais longa:
Data/hora de início: 2024-05-10 17:33:00
Distância: 86789.20 milhas


# Questão 5 — Quantidade de viagens por hora

A hora é extraída de `tpep_pickup_datetime`.

### MAP
Emitimos:

```text
hora    1
```

### SHUFFLE
Agrupamos as ocorrências por hora.

### REDUCE
Somamos a quantidade de viagens de cada hora.

In [9]:
def map_viagens_por_hora(nome_arquivo_entrada, nome_arquivo_saida):
    with open(nome_arquivo_entrada, "r", encoding="utf-8", newline="") as entrada, \
         open(nome_arquivo_saida, "w", encoding="utf-8") as saida:

        leitor = csv.DictReader(entrada)

        for linha in leitor:
            data_hora = linha["tpep_pickup_datetime"].strip()

            if data_hora == "":
                continue

            hora = datetime.strptime(data_hora, "%Y-%m-%d %H:%M:%S").hour
            saida.write(f"{hora:02d}\t1\n")


map_viagens_por_hora(ARQUIVO, "q5_mapper.txt")
shuffle("q5_mapper.txt", "q5_shuffle.txt")
resultado_q5 = reducer_soma_inteiros("q5_shuffle.txt", "q5_resultado.txt")

print("Quantidade de viagens por hora:")
for hora, quantidade in sorted(resultado_q5.items()):
    print(f"{hora}:00 -> {quantidade}")

Quantidade de viagens por hora:
00:00 -> 29165
01:00 -> 18822
02:00 -> 12280
03:00 -> 8281
04:00 -> 6054
05:00 -> 6194
06:00 -> 13966
07:00 -> 28065
08:00 -> 38308
09:00 -> 42309
10:00 -> 44804
11:00 -> 48295
12:00 -> 53128
13:00 -> 55362
14:00 -> 59345
15:00 -> 60205
16:00 -> 61563
17:00 -> 67880
18:00 -> 71403
19:00 -> 62752
20:00 -> 56542
21:00 -> 58333
22:00 -> 54581
23:00 -> 42363


# Questão 6 — Distância total percorrida por hora

### MAP
Emitimos:

```text
hora    trip_distance
```

### SHUFFLE
Agrupamos as distâncias por hora.

### REDUCE
Somamos as distâncias de cada hora.

In [10]:
def reducer_soma_float(nome_arquivo_entrada, nome_arquivo_saida):
    with open(nome_arquivo_entrada, "r", encoding="utf-8") as arquivo:
        grupos = ast.literal_eval(arquivo.read())

    resultado = {}

    for chave, valores in grupos.items():
        resultado[chave] = reduce(
            lambda x, y: x + y,
            map(float, valores)
        )

    with open(nome_arquivo_saida, "w", encoding="utf-8") as saida:
        for chave in sorted(resultado):
            saida.write(
                f"{chave}\t{resultado[chave]:.2f}\n"
            )

    return resultado

In [11]:
def map_distancia_por_hora(nome_arquivo_entrada, nome_arquivo_saida):
    with open(nome_arquivo_entrada, "r", encoding="utf-8", newline="") as entrada, \
         open(nome_arquivo_saida, "w", encoding="utf-8") as saida:

        leitor = csv.DictReader(entrada)

        for linha in leitor:
            data_hora = linha["tpep_pickup_datetime"].strip()
            distancia = linha["trip_distance"].strip()

            if data_hora == "" or distancia == "":
                continue

            hora = datetime.strptime(data_hora, "%Y-%m-%d %H:%M:%S").hour
            saida.write(f"{hora:02d}\t{float(distancia)}\n")


map_distancia_por_hora(ARQUIVO, "q6_mapper.txt")
shuffle("q6_mapper.txt", "q6_shuffle.txt")
resultado_q6 = reducer_soma_float("q6_shuffle.txt", "q6_resultado.txt")

print("Distância total percorrida por hora:")
for hora, distancia in sorted(resultado_q6.items()):
    print(f"{hora}:00 -> {distancia:.2f} milhas")

Distância total percorrida por hora:
00:00 -> 109568.73 milhas
01:00 -> 60695.19 milhas
02:00 -> 36643.43 milhas
03:00 -> 29745.90 milhas
04:00 -> 28350.30 milhas
05:00 -> 91136.53 milhas
06:00 -> 131056.86 milhas
07:00 -> 195237.07 milhas
08:00 -> 169111.73 milhas
09:00 -> 222652.02 milhas
10:00 -> 138342.76 milhas
11:00 -> 145189.30 milhas
12:00 -> 166186.16 milhas
13:00 -> 190847.83 milhas
14:00 -> 215732.27 milhas
15:00 -> 312374.38 milhas
16:00 -> 238718.29 milhas
17:00 -> 321659.21 milhas
18:00 -> 252601.19 milhas
19:00 -> 265021.98 milhas
20:00 -> 267461.66 milhas
21:00 -> 283775.08 milhas
22:00 -> 189960.49 milhas
23:00 -> 161753.45 milhas


# Resumo dos resultados

A célula abaixo reúne os resultados finais calculados pelas etapas MapReduce.

In [12]:
print("=" * 65)
print("1) NÚMERO DE VIAGENS POR TIPO DE PAGAMENTO")
for tipo, quantidade in sorted(resultado_q1.items()):
    print(f"   {tipo}: {quantidade}")

print("\n2) RECEITA TOTAL POR TIPO DE PAGAMENTO")
for tipo, receita in sorted(resultado_q2.items()):
    print(f"   {tipo}: $ {receita:.2f}")

print(f"\n3) TARIFA MÉDIA: $ {resultado_q3:.4f}")

print("\n4) VIAGEM MAIS LONGA")
print(f"   Início: {data_hora_q4}")
print(f"   Distância: {distancia_q4:.2f} milhas")

print("\n5) QUANTIDADE DE VIAGENS POR HORA")
for hora, quantidade in sorted(resultado_q5.items()):
    print(f"   {hora}:00 -> {quantidade}")

print("\n6) DISTÂNCIA TOTAL PERCORRIDA POR HORA")
for hora, distancia in sorted(resultado_q6.items()):
    print(f"   {hora}:00 -> {distancia:.2f} milhas")

print("=" * 65)

1) NÚMERO DE VIAGENS POR TIPO DE PAGAMENTO
   Cash: 136221
   Credit card: 743405
   Dispute: 16543
   Flex Fare trip: 97124
   No charge: 6707

2) RECEITA TOTAL POR TIPO DE PAGAMENTO
   Cash: $ 3168095.90
   Credit card: $ 21785219.95
   Dispute: $ 25214.51
   Flex Fare trip: $ 2376069.77
   No charge: $ 53932.48

3) TARIFA MÉDIA: $ 18.8603

4) VIAGEM MAIS LONGA
   Início: 2024-05-10 17:33:00
   Distância: 86789.20 milhas

5) QUANTIDADE DE VIAGENS POR HORA
   00:00 -> 29165
   01:00 -> 18822
   02:00 -> 12280
   03:00 -> 8281
   04:00 -> 6054
   05:00 -> 6194
   06:00 -> 13966
   07:00 -> 28065
   08:00 -> 38308
   09:00 -> 42309
   10:00 -> 44804
   11:00 -> 48295
   12:00 -> 53128
   13:00 -> 55362
   14:00 -> 59345
   15:00 -> 60205
   16:00 -> 61563
   17:00 -> 67880
   18:00 -> 71403
   19:00 -> 62752
   20:00 -> 56542
   21:00 -> 58333
   22:00 -> 54581
   23:00 -> 42363

6) DISTÂNCIA TOTAL PERCORRIDA POR HORA
   00:00 -> 109568.73 milhas
   01:00 -> 60695.19 milhas
   02:00 -> 

## Conclusão

A solução foi estruturada explicitamente em **Map → Shuffle → Reduce**, conforme solicitado na atividade. Cada questão possui um mapper responsável pela emissão dos pares chave/valor, o shuffle agrupa os valores pela chave e o reducer produz o resultado agregado.

Os arquivos `q*_mapper.txt`, `q*_shuffle.txt` e `q*_resultado.txt` também permitem observar fisicamente o que acontece em cada etapa do processamento.